In [64]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

order_payments = pd.read_csv(RAW/ 'olist_order_payments_dataset.csv')

In [65]:
order_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [66]:
order_payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [67]:
order_payments.head(30)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
5,298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
6,771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
7,3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
8,1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
9,0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [68]:
order_payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [69]:
order_payments['payment_installments'].max()

np.int64(24)

In [70]:
order_payments.duplicated(subset=['order_id','payment_sequential']).sum()

np.int64(0)

In [71]:
order_payments['order_id'].nunique()

99440

In [72]:
order_payments[['payment_installments','payment_value']].describe()

,payment_installments,payment_value
count,103886.000000,103886.000000
mean,2.853349,154.100380
std,2.687051,217.494064
min,0.000000,0.000000
25%,1.000000,56.790000
50%,1.000000,100.000000
75%,4.000000,171.837500
max,24.000000,13664.080000


In [73]:
(order_payments['payment_installments']< 1).sum()

np.int64(2)

In [74]:
order_payments[order_payments['payment_value'] <= 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [75]:
order_payments[order_payments['payment_installments'] < 1]      

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [76]:
order_payments[order_payments['payment_type'] == 'not_defined']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [77]:
order_payments['has_data_issue'] = (order_payments['payment_installments'] < 1) | (order_payments['payment_value'] == 0)

In [78]:
order_payments['has_data_issue'].value_counts()      # expect True 11 / False 103,875

has_data_issue
False    103875
True         11
Name: count, dtype: int64

In [79]:
orders = pd.read_csv(CLEAN / 'orders_clean.csv', usecols=['order_id','order_status'])

In [80]:
order_payments['order_id'].isin(orders['order_id']).value_counts()


order_id
True    103886
Name: count, dtype: int64

In [81]:
no_pay = ~orders['order_id'].isin(order_payments['order_id'])

In [82]:
orders.loc[no_pay]

,order_id,order_status
30710,bfbd0f9bdef84302105ad712db648a6c,delivered


In [83]:
nd = order_payments.loc[order_payments['payment_type'] == 'not_defined', 'order_id']

In [84]:
orders[orders['order_id'].isin(nd)]

,order_id,order_status
1130,00b1cb0320190ca0daa2c88b35206009,canceled
39919,4637ca194b6387e2d538dc89b124b0ee,canceled
40235,c8c528189310eaa44a745b8d9d26908b,canceled


In [85]:
items = pd.read_csv(CLEAN/ 'order_items_clean.csv', usecols=['order_id','item_total'])

In [86]:
items_total = items.groupby('order_id')['item_total'].sum().rename('items_total')
paid_total = order_payments.groupby('order_id')['payment_value'].sum().rename('paid_total')

In [87]:
rec = pd.concat([items_total,paid_total], axis=1)
rec.isna().sum()

items_total    775
paid_total       1
dtype: int64

In [88]:
both = rec.dropna().copy()
both['diff'] = (both['paid_total'] - both['items_total']).round(2)

In [89]:
len(both)

98665

In [90]:
(both['diff'].abs() > 0.01).sum()

np.int64(303)

In [91]:
both['diff'].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: diff, dtype: float64

In [92]:
rec.loc[rec['items_total'].isna(), 'paid_total'].sum().round(2)

np.float64(162591.95)

In [93]:
pt = order_payments.groupby('order_id')['payment_type'].agg(lambda s: '+'.join(sorted(s.unique())))
mism = both[both['diff'].abs() > 0.01]
mism.join(pt)['payment_type'].value_counts().head()

payment_type
credit_card            275
boleto                  14
credit_card+voucher      7
debit_card               7
Name: count, dtype: int64

In [94]:
order_payments.to_csv(CLEAN / 'order_payments_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'order_payments_clean.csv')
print(check.shape)                                                                       # expect (103886, 6)
print(check.duplicated(subset=['order_id', 'payment_sequential']).sum())                 # expect 0
print(check['has_data_issue'].value_counts())                                            # True 11 / False 103,875
print(check['payment_value'].sum().round(2))                                             # total value, for the README

(103886, 6)
0
has_data_issue
False    103875
True         11
Name: count, dtype: int64
16008872.12


In [ ]:
sell